# 070 CASE 060 — Vegetation edge

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/imintengine/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_060-Vegetation-Edge.ipynb)

**Purpose:** Map the boundary between vegetated and non-vegetated zones — useful for tracking clear-cuts, re-growth, dune zonation and wetland delineation.

**Partners:**
- **Skogsstyrelsen** — harvesting notifications, re-forestation tracking
- **Naturvårdsverket** — NMD reference, wetland inventory
- **Stockholm University (Physical Geography)** — VedgeSat method validation
- **RISE** — ImintEngine implementation

**Method reference:** Muir et al. (2024), *Earth Surf. Process. Landf.* 49, 2405–2423; Nugraha et al. (2026), *Front. Mar. Sci.* 13, 1757991.

**Licence:** CC0 1.0 notebook.

## 1. Method

NDVI-based thresholding pipeline:

```
Sentinel-2 → NDVI = (NIR − Red) / (NIR + Red)   [B08, B04]
           → Otsu / Weighted-Peaks / fixed threshold
           → 3-class map: water / non-vegetated / vegetated
           → morphological cleaning
           → vegetation-edge centreline mask
           → contour count
```

`VegetationEdgeAnalyzer` is in `ANALYZER_REGISTRY` and runs via `run_job()`.

**Verified outputs:** `segmentation_map`, `vegetation_edge_mask`, `vegetation_fraction`, `class_fractions`, `ndvi`, `n_contours`.

## 2. Setup

In [ ]:
from executors.local import LocalExecutor
from imint.engine import run_job
import matplotlib.pyplot as plt
import folium

def get_outputs(result, name):
    for ar in result.analyzer_results:
        if ar.analyzer == name and ar.success:
            return ar.outputs
    return None

# Småland coastal mosaic — forest, open land, wetland mix
AOI = {"west": 16.30, "south": 56.95, "east": 16.50, "north": 57.05}
DATE = "2024-07-15"  # peak vegetation

print(f"AOI: {AOI}")
print(f"Date: {DATE}")

## 3. Run analysis

In [ ]:
executor = LocalExecutor(
    output_dir="outputs/vegetationskant",
    config_path="config/analyzers.yaml",
)

job = executor.build_job(date=DATE, coords=AOI)
result = run_job(job)

ve = get_outputs(result, "vegetation_edge")
if ve is not None:
    print(f"vegetation_fraction: {ve['vegetation_fraction']:.3f}")
    print(f"n_contours:          {ve['n_contours']}")
    print(f"class_fractions:")
    for cls, frac in ve["class_fractions"].items():
        print(f"  {cls}: {frac*100:.1f}%")
else:
    print("vegetation_edge analyzer unavailable.")

## 4. Visualize

In [ ]:
center = [(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2]
m = folium.Map(location=center, zoom_start=11, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satellite",
).add_to(m)
folium.Rectangle(
    bounds=[[AOI["south"], AOI["west"]], [AOI["north"], AOI["east"]]],
    color="#00aa00", weight=2, fill=False,
).add_to(m)
folium.LayerControl().add_to(m)
m

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0, 0].imshow(job.rgb)
axes[0, 0].set_title(f"RGB ({DATE})")
axes[0, 0].axis("off")

if ve is not None:
    im = axes[0, 1].imshow(ve["ndvi"], cmap="RdYlGn", vmin=-0.2, vmax=0.9)
    axes[0, 1].set_title("NDVI")
    axes[0, 1].axis("off")
    plt.colorbar(im, ax=axes[0, 1], fraction=0.046)

    axes[1, 0].imshow(ve["segmentation_map"], cmap="tab10", vmin=0, vmax=9)
    axes[1, 0].set_title("3-class: water / non-veg / vegetated")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(ve["vegetation_edge_mask"], cmap="binary")
    axes[1, 1].set_title("Vegetation-edge centreline")
    axes[1, 1].axis("off")
else:
    for ax in axes.flat[1:]:
        ax.text(0.5, 0.5, "vegetation_edge unavailable", ha="center", va="center")
        ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Interpretation

**Use cases:**
- **Forestry:** track clear-cut extents and re-forestation progress
- **Coastal research:** dune-vegetation advance/retreat (SU collaboration)
- **Wetland management:** reed-belt vs open water boundaries
- **Agriculture:** arable-field vs forest boundary

**Method vs VedgeSat:**
- Same NDVI approach; Weighted-Peaks threshold from Muir et al. (2024)
- ImintEngine adds morphological cleaning + single-pass vectorisation

**Next:**
- Multi-temporal edge comparison → detect clear-cuts / re-growth
- Cross-reference with Skogsstyrelsen's harvest notifications
- Combine with NDWI for better wetland delineation

### Links
- [`imint/analyzers/vegetation_edge.py`](https://github.com/TobiasEdman/imintengine/blob/main/imint/analyzers/vegetation_edge.py)
- [VedgeSat (Muir et al., 2024)](https://doi.org/10.1002/esp.5835)
- [Nugraha et al. (2026)](https://doi.org/10.3389/fmars.2026.1757991)